# Диагностика полей по моделям во всей витрине

Notebook предназначен для поиска физического расположения значений в Библиотеке моделей.

Что делает:
- принимает список `MODEL_SIDS`, `MODEL_CODES` и/или `MODEL_NAMES`;
- находит **все версии** выбранных моделей;
- расширяет область поиска на связанные сущности: промверсии, валидации, мониторинги, результаты мониторинга, выборки, внедрения, бизнес-задачи, исследования и проекты;
- проходит по всем таблицам `SOURCE_DB`, доступным в Spark catalog;
- для каждой таблицы, которую можно связать с выбранными моделями, получает до **10 различных непустых значений каждого столбца**;
- отдельно собирает потенциальные ссылки/репозитории/Bitbucket-значения и целые строки, в которых они встречаются;
- пишет результат в один Excel.

Важно: для этой диагностики используется **вся история записей**, без отсечения по `START_DT/END_DT`. Это специально: мы ищем, где и в каком виде физически могло храниться значение.

In [ ]:
import os
import re
import sys
from functools import reduce
from pathlib import Path

SPARK_CLIENT = Path("/usr/sdp/current/spark3.5.1-client")

if SPARK_CLIENT.exists():
    os.environ.setdefault("SPARK_MAJOR_VERSION", "3.5.1")
    os.environ.setdefault("SPARK_HOME", str(SPARK_CLIENT))
    os.environ.setdefault("PYSPARK_DRIVER_PYTHON", sys.executable)

    spark_python = str(SPARK_CLIENT / "python")
    if spark_python not in sys.path:
        sys.path.insert(0, spark_python)

    py4j_archives = sorted(
        (SPARK_CLIENT / "python/lib").glob("py4j-*-src.zip")
    )
    if py4j_archives:
        py4j_path = str(py4j_archives[-1])
        if py4j_path not in sys.path:
            sys.path.insert(0, py4j_path)

from pyspark import SparkConf, StorageLevel
from pyspark.sql import SparkSession, Window, functions as F

from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill

# Значения вроде $not_applicable$ сохраняются в VALUES_10,
# потому что для диагностики нам важно видеть реальные сырые значения.
SERVICE_TEXT = {
    "",
    "null",
    "none",
    "nan",
}

# Для отдельного листа LINK_HITS.
LINK_VALUE_REGEX = (
    r"(?i)("
    r"bitbucket|"
    r"https?://|"
    r"git@|"
    r"\.git(?:$|[\s/?#])|"
    r"repository|"
    r"repo(?:sitory)?|"
    r"репозитор|"
    r"ссылк|"
    r"url"
    r")"
)

LINK_COLUMN_REGEX = re.compile(
    r"(?i)("
    r"link|url|uri|repo|repstr|repository|"
    r"branch|commit|git|bitbucket|"
    r"path|ссыл|репоз"
    r")"
)


## Параметры запуска

In [ ]:
# ============================================================
# ПАРАМЕТРЫ ЗАПУСКА
# ============================================================

SOURCE_DB = "prx_pri_custom_ris_l_library_custom_risk_model_library"

# Можно заполнить один или несколько списков.
MODEL_SIDS = [
    # "113808",
]

MODEL_CODES = [
    # "MODEL_CODE",
]

MODEL_NAMES = [
    # "Точное наименование модели",
]

# До 10 разных значений по каждому полю каждой связанной таблицы.
VALUES_PER_COLUMN = 10

# До 50 полных строк с потенциальными ссылками по каждой таблице.
LINK_ROWS_PER_TABLE = 50

# Итоговый Excel.
OUTPUT_FILE = "model_mart_probe.xlsx"

print("Параметры заданы")


## 1. Spark

In [ ]:
spark = SparkSession.getActiveSession()

if spark is None:
    conf = (
        SparkConf()
        .setAppName("model_mart_probe")
        .setMaster("yarn")
        .set("spark.executor.cores", "2")
        .set("spark.executor.memory", "6g")
        .set("spark.executor.memoryOverhead", "1g")
        .set("spark.driver.maxResultSize", "4g")
        .set("spark.shuffle.service.enabled", "true")
        .set("spark.dynamicAllocation.enabled", "true")
        .set("spark.dynamicAllocation.initialExecutors", "4")
        .set("spark.dynamicAllocation.maxExecutors", "12")
        .set("spark.port.maxRetries", "150")
    )
    spark = (
        SparkSession.builder
        .config(conf=conf)
        .enableHiveSupport()
        .getOrCreate()
    )

spark.sparkContext.setLogLevel("ERROR")

print("Python:", sys.version)
print("Spark:", spark.version)
print("DB:", SOURCE_DB)


## 2. Вспомогательные функции

In [ ]:
def table_exists(table):
    return spark.catalog.tableExists(
        "{}.{}".format(SOURCE_DB, table)
    )


def read_table(table):
    full_name = "{}.{}".format(SOURCE_DB, table)
    if not table_exists(table):
        raise RuntimeError("Не найдена таблица: {}".format(full_name))

    df = spark.table(full_name)
    lower = [c.lower() for c in df.columns]

    if len(lower) != len(set(lower)):
        raise RuntimeError(
            "{}: после lower() есть дубли имён колонок".format(
                full_name
            )
        )

    return df.toDF(*lower)


def as_string_set(values):
    return {
        str(v).strip()
        for v in values
        if v is not None and str(v).strip() != ""
    }


def collect_ids(
    table,
    filter_column,
    filter_values,
    target_column,
):
    """
    Возвращает distinct target IDs из всей истории таблицы.
    Если таблицы/колонки нет, возвращает пустое множество.
    """
    filter_values = as_string_set(filter_values)
    if not filter_values or not table_exists(table):
        return set()

    df = read_table(table)
    cols = set(df.columns)

    if filter_column not in cols or target_column not in cols:
        return set()

    rows = (
        df.filter(
            F.col(filter_column)
            .cast("string")
            .isin(sorted(filter_values))
        )
        .select(
            F.col(target_column)
            .cast("string")
            .alias("__id")
        )
        .filter(
            F.col("__id").isNotNull()
            & (F.trim(F.col("__id")) != "")
        )
        .distinct()
        .collect()
    )

    return {r["__id"] for r in rows}


def union_collect(specs):
    result = set()
    for table, filter_col, filter_values, target_col in specs:
        result |= collect_ids(
            table,
            filter_col,
            filter_values,
            target_col,
        )
    return result


def meaningful_value(col):
    text = F.trim(col.cast("string"))
    return (
        col.isNotNull()
        & (~F.lower(text).isin(sorted(SERVICE_TEXT)))
    )


def latest_display(df, keys):
    """
    Только для листа SCOPE: выбирает одно наиболее позднее техническое
    состояние сущности. На сам поиск по витрине это не влияет.
    """
    order = []

    if "start_dt" in df.columns:
        order.append(F.col("start_dt").desc_nulls_last())
    if "ctl_datechange" in df.columns:
        order.append(F.col("ctl_datechange").desc_nulls_last())
    if "ctl_loading" in df.columns:
        order.append(
            F.col("ctl_loading").cast("long").desc_nulls_last()
        )

    if not order:
        return df.dropDuplicates(keys)

    w = Window.partitionBy(*keys).orderBy(*order)
    return (
        df.withColumn("__rn", F.row_number().over(w))
        .filter(F.col("__rn") == 1)
        .drop("__rn")
    )


## 3. Формирование области поиска

In [ ]:
# ------------------------------------------------------------
# 1. Разрешаем список моделей
# ------------------------------------------------------------
requested_sids = as_string_set(MODEL_SIDS)
requested_codes = as_string_set(MODEL_CODES)
requested_names = as_string_set(MODEL_NAMES)

if not (requested_sids or requested_codes or requested_names):
    raise ValueError(
        "Заполни MODEL_SIDS, MODEL_CODES и/или MODEL_NAMES"
    )

models_raw = read_table("t_model")

model_filter = F.lit(False)

if requested_sids:
    model_filter = model_filter | (
        F.col("model_sid").cast("string").isin(
            sorted(requested_sids)
        )
    )

if requested_codes:
    model_filter = model_filter | (
        F.col("model_code").cast("string").isin(
            sorted(requested_codes)
        )
    )

if requested_names:
    model_filter = model_filter | (
        F.col("model_name").cast("string").isin(
            sorted(requested_names)
        )
    )

selected_models_raw = models_raw.filter(model_filter)

resolved_model_sids = {
    r["model_sid"]
    for r in (
        selected_models_raw
        .select(
            F.col("model_sid").cast("string").alias("model_sid")
        )
        .filter(F.col("model_sid").isNotNull())
        .distinct()
        .collect()
    )
}

# Не теряем вручную указанный SID даже если T_MODEL по какой-то
# причине не вернул его в текущем доступном срезе.
resolved_model_sids |= requested_sids

if not resolved_model_sids:
    raise RuntimeError(
        "Ни одной модели по заданному списку не найдено"
    )

# ------------------------------------------------------------
# 2. Все версии выбранных моделей
# ------------------------------------------------------------
versions_raw = read_table("t_model_ver")

selected_versions_raw = versions_raw.filter(
    F.col("model_sid")
    .cast("string")
    .isin(sorted(resolved_model_sids))
)

model_ver_sids = {
    r["model_ver_sid"]
    for r in (
        selected_versions_raw
        .select(
            F.col("model_ver_sid")
            .cast("string")
            .alias("model_ver_sid")
        )
        .filter(F.col("model_ver_sid").isNotNull())
        .distinct()
        .collect()
    )
}

if not model_ver_sids:
    raise RuntimeError(
        "У выбранных моделей не найдено ни одной версии"
    )

# ------------------------------------------------------------
# 3. Расширяем область поиска на связанные сущности
# ------------------------------------------------------------

# Промверсии
model_ver_prom_sids = union_collect([
    (
        "t_model_ver_prom",
        "model_ver_sid",
        model_ver_sids,
        "model_ver_prom_sid",
    ),
    (
        "t_model_ver_anlt_dtl",
        "model_ver_sid",
        model_ver_sids,
        "model_ver_prom_sid",
    ),
])

# Валидации
valid_sids = collect_ids(
    "t_valid",
    "model_ver_sid",
    model_ver_sids,
    "valid_sid",
)

# Мониторинги
montrg_auto_sids = collect_ids(
    "t_model_ver_x_montrg_auto",
    "model_ver_sid",
    model_ver_sids,
    "montrg_auto_sid",
)

montrg_manual_sids = collect_ids(
    "t_model_ver_x_montrg_manual",
    "model_ver_sid",
    model_ver_sids,
    "montrg_manual_sid",
)

# Результаты мониторингов
montrg_auto_rslt_sids = collect_ids(
    "t_montrg_auto_rslt",
    "montrg_auto_sid",
    montrg_auto_sids,
    "montrg_auto_rslt_sid",
)

montrg_manual_rslt_sids = collect_ids(
    "t_montrg_manual_rslt",
    "montrg_manual_sid",
    montrg_manual_sids,
    "montrg_manual_rslt_sid",
)

# Внедрения
prom_implm_sids = collect_ids(
    "t_model_ver_prom_x_prom_implm",
    "model_ver_prom_sid",
    model_ver_prom_sids,
    "prom_implm_sid",
)

pilot_implm_sids = collect_ids(
    "t_model_ver_prom_x_pilot_implm",
    "model_ver_prom_sid",
    model_ver_prom_sids,
    "pilot_implm_sid",
)

# Бизнес-задачи -> исследования
busn_task_sids = collect_ids(
    "t_model_ver_x_busn_task",
    "model_ver_sid",
    model_ver_sids,
    "busn_task_sid",
)

research_sids = collect_ids(
    "t_research",
    "busn_task_sid",
    busn_task_sids,
    "research_sid",
)

# Выборки: собираем через все возможные связи
sample_data_sids = union_collect([
    (
        "t_sample_data",
        "model_ver_sid",
        model_ver_sids,
        "sample_data_sid",
    ),
    (
        "t_sample_data",
        "valid_sid",
        valid_sids,
        "sample_data_sid",
    ),
    (
        "t_sample_data",
        "montrg_auto_rslt_sid",
        montrg_auto_rslt_sids,
        "sample_data_sid",
    ),
    (
        "t_sample_data",
        "montrg_manual_rslt_sid",
        montrg_manual_rslt_sids,
        "sample_data_sid",
    ),
    (
        "t_sample_data",
        "res_sid",
        research_sids,
        "sample_data_sid",
    ),
])

# Проекты SberDS
proj_sids = union_collect([
    (
        "t_proj_rslt_rb_corp",
        "model_ver_sid",
        model_ver_sids,
        "proj_sid",
    ),
    (
        "t_proj_test_rb_corp",
        "model_ver_sid",
        model_ver_sids,
        "proj_sid",
    ),
    (
        "t_proj_test_rslt_eco",
        "model_ver_sid",
        model_ver_sids,
        "proj_sid",
    ),
    (
        "t_proj_rslt_rb_corp",
        "valid_sid",
        valid_sids,
        "proj_sid",
    ),
])

# Для универсального ENT_SID ищем среди всех известных сущностей.
entity_sids = (
    resolved_model_sids
    | model_ver_sids
    | model_ver_prom_sids
    | valid_sids
    | montrg_auto_sids
    | montrg_manual_sids
    | montrg_auto_rslt_sids
    | montrg_manual_rslt_sids
    | sample_data_sids
    | prom_implm_sids
    | pilot_implm_sids
    | busn_task_sids
    | research_sids
    | proj_sids
)

SCOPE_BY_KEY = {
    "model_sid": resolved_model_sids,
    "model_ver_sid": model_ver_sids,
    "model_ver_prom_sid": model_ver_prom_sids,
    "valid_sid": valid_sids,
    "montrg_auto_sid": montrg_auto_sids,
    "montrg_manual_sid": montrg_manual_sids,
    "montrg_auto_rslt_sid": montrg_auto_rslt_sids,
    "montrg_manual_rslt_sid": montrg_manual_rslt_sids,
    "sample_data_sid": sample_data_sids,
    "prom_implm_sid": prom_implm_sids,
    "pilot_implm_sid": pilot_implm_sids,
    "busn_task_sid": busn_task_sids,
    "research_sid": research_sids,
    "res_sid": research_sids,
    "proj_sid": proj_sids,
    "ent_sid": entity_sids,
}

print("Моделей:", len(resolved_model_sids))
print("Версий:", len(model_ver_sids))
print("Промверсий:", len(model_ver_prom_sids))
print("Валидаций:", len(valid_sids))
print("Автомониторингов:", len(montrg_auto_sids))
print("Ручных мониторингов:", len(montrg_manual_sids))
print("Выборок:", len(sample_data_sids))
print("Бизнес-задач:", len(busn_task_sids))
print("Исследований:", len(research_sids))
print("Проектов:", len(proj_sids))


In [ ]:
# Модели для листа SCOPE
models_scope = (
    latest_display(
        selected_models_raw,
        ["model_sid"],
    )
    .select(
        F.col("model_sid").cast("string").alias("model_sid"),
        "model_code",
        "model_name",
    )
)

# Все версии для листа SCOPE
versions_scope_cols = [
    c for c in (
        "model_sid",
        "model_ver_sid",
        "model_ver_num",
        "model_ver_name",
        "model_ver_stts_name",
        "model_ver_mlstorage_branch_link_txt",
        "start_dt",
        "end_dt",
        "ctl_datechange",
        "ctl_loading",
    )
    if c in selected_versions_raw.columns
]

versions_scope = (
    latest_display(
        selected_versions_raw.select(*versions_scope_cols),
        ["model_ver_sid"],
    )
)

print("Выбранные модели:")
models_scope.show(100, truncate=False)

print("Все найденные версии:")
versions_scope.show(200, truncate=False)


## 4. Сканирование всех таблиц витрины

In [ ]:
def filter_table_to_scope(table):
    """
    Возвращает:
      filtered_df,
      список ключей, по которым таблица привязана к области поиска.

    Если в таблице несколько известных ключей, используется OR:
    строка относится к выборке, если совпал хотя бы один ключ.
    """
    df = read_table(table)
    cols = set(df.columns)

    conditions = []
    used_keys = []

    for key, values in SCOPE_BY_KEY.items():
        if key not in cols or not values:
            continue

        conditions.append(
            F.col(key)
            .cast("string")
            .isin(sorted(values))
        )
        used_keys.append(key)

    if not conditions:
        return None, []

    condition = reduce(lambda a, b: a | b, conditions)
    return df.filter(condition), used_keys


def make_long_values(df, table):
    """
    Переводит всю связанную выборку таблицы в long-format:
    table_name | column_name | value
    """
    structs = []

    for c in df.columns:
        structs.append(
            F.struct(
                F.lit(c).alias("column_name"),
                F.col(c).cast("string").alias("value"),
            )
        )

    if not structs:
        return None

    return (
        df.select(F.explode(F.array(*structs)).alias("__kv"))
        .select(
            F.lit(table).alias("table_name"),
            F.col("__kv.column_name").alias("column_name"),
            F.col("__kv.value").alias("value"),
        )
        .filter(
            F.col("value").isNotNull()
            & (F.trim(F.col("value")) != "")
        )
    )


def collect_values10(long_df):
    w = Window.partitionBy(
        "table_name",
        "column_name",
    ).orderBy(F.col("value"))

    return (
        long_df.dropDuplicates(
            ["table_name", "column_name", "value"]
        )
        .withColumn(
            "value_no",
            F.row_number().over(w),
        )
        .filter(F.col("value_no") <= VALUES_PER_COLUMN)
        .select(
            "table_name",
            "column_name",
            "value_no",
            "value",
        )
    )


def collect_link_hits(long_df):
    name_match = F.lower(F.col("column_name")).rlike(
        LINK_COLUMN_REGEX.pattern
    )

    value_match = F.col("value").rlike(LINK_VALUE_REGEX)

    # Для link-подобных колонок не считаем $not_applicable$
    # и аналогичные технические пустышки интересным hit.
    lower_value = F.lower(F.trim(F.col("value")))
    non_service = ~lower_value.isin(
        sorted(SERVICE_TEXT | {"$not_applicable$"})
    )

    hits = (
        long_df.filter(
            value_match | (name_match & non_service)
        )
        .dropDuplicates(
            ["table_name", "column_name", "value"]
        )
    )

    w = Window.partitionBy(
        "table_name",
        "column_name",
    ).orderBy(F.col("value"))

    return (
        hits.withColumn(
            "hit_no",
            F.row_number().over(w),
        )
        .filter(F.col("hit_no") <= 100)
        .select(
            "table_name",
            "column_name",
            "hit_no",
            "value",
        )
    )


def collect_link_rows(df, table):
    """
    Полные строки, где:
    - значение само похоже на URL/репозиторий/Bitbucket;
    - либо заполнено поле с link/repo/branch/commit-подобным именем.
    """
    row_conditions = []

    for c in df.columns:
        text = F.col(c).cast("string")
        lower_text = F.lower(F.trim(text))

        value_like = text.rlike(LINK_VALUE_REGEX)

        if LINK_COLUMN_REGEX.search(c):
            name_like = (
                meaningful_value(F.col(c))
                & (lower_text != "$not_applicable$")
            )
        else:
            name_like = F.lit(False)

        row_conditions.append(value_like | name_like)

    if not row_conditions:
        return None

    condition = reduce(lambda a, b: a | b, row_conditions)

    return (
        df.filter(condition)
        .select(
            F.lit(table).alias("table_name"),
            F.substring(
                F.to_json(F.struct(*[F.col(c) for c in df.columns])),
                1,
                30000,
            ).alias("row_json"),
        )
        .limit(LINK_ROWS_PER_TABLE)
    )


catalog_tables = sorted({
    t.name.lower()
    for t in spark.catalog.listTables(SOURCE_DB)
})

print("Таблиц в catalog:", len(catalog_tables))

coverage_rows = []
values_dfs = []
hits_dfs = []
link_rows_dfs = []

for i, table in enumerate(catalog_tables, 1):
    print(
        "[{}/{}] {}".format(
            i,
            len(catalog_tables),
            table,
        )
    )

    try:
        filtered, used_keys = filter_table_to_scope(table)

        if filtered is None:
            coverage_rows.append({
                "table_name": table,
                "status": "NO_RELATION_KEY",
                "relation_keys": "",
                "has_rows": 0,
                "error": "",
            })
            continue

        has_rows = filtered.limit(1).count()

        if not has_rows:
            coverage_rows.append({
                "table_name": table,
                "status": "NO_MATCH",
                "relation_keys": ", ".join(used_keys),
                "has_rows": 0,
                "error": "",
            })
            continue

        filtered = filtered.persist(
            StorageLevel.MEMORY_AND_DISK
        )

        long_df = make_long_values(filtered, table)

        if long_df is not None:
            long_df = long_df.persist(
                StorageLevel.MEMORY_AND_DISK
            )

            values_dfs.append(
                collect_values10(long_df)
            )

            hits_dfs.append(
                collect_link_hits(long_df)
            )

        rows_df = collect_link_rows(filtered, table)
        if rows_df is not None:
            link_rows_dfs.append(rows_df)

        coverage_rows.append({
            "table_name": table,
            "status": "OK",
            "relation_keys": ", ".join(used_keys),
            "has_rows": 1,
            "error": "",
        })

        if long_df is not None:
            long_df.unpersist()
        filtered.unpersist()

    except Exception as exc:
        coverage_rows.append({
            "table_name": table,
            "status": "ERROR",
            "relation_keys": "",
            "has_rows": 0,
            "error": "{}: {}".format(
                type(exc).__name__,
                str(exc)[:1000],
            ),
        })


def union_all(dfs):
    if not dfs:
        return None

    result = dfs[0]
    for df in dfs[1:]:
        result = result.unionByName(
            df,
            allowMissingColumns=True,
        )
    return result


coverage = spark.createDataFrame(coverage_rows)

values10 = union_all(values_dfs)
link_hits = union_all(hits_dfs)
link_rows = union_all(link_rows_dfs)

print("\nCoverage:")
coverage.groupBy("status").count().orderBy("status").show(
    100,
    truncate=False,
)

print("\nПотенциальные поля со ссылками:")
if link_hits is not None:
    link_hits.orderBy(
        "table_name",
        "column_name",
        "hit_no",
    ).show(200, truncate=False)
else:
    print("Не найдено")


## 5. Выгрузка Excel

In [ ]:
def spark_rows(df, columns):
    if df is None:
        return
    for row in df.select(*columns).toLocalIterator():
        yield [
            row[c]
            for c in columns
        ]


def add_sheet(wb, title, columns, rows):
    ws = wb.create_sheet(title=title)

    header_fill = PatternFill(
        fill_type="solid",
        fgColor="D9EAF7",
    )
    header_font = Font(bold=True)

    for j, col in enumerate(columns, 1):
        cell = ws.cell(row=1, column=j, value=col)
        cell.fill = header_fill
        cell.font = header_font

    row_idx = 2
    for values in rows:
        for col_idx, value in enumerate(values, 1):
            ws.cell(
                row=row_idx,
                column=col_idx,
                value=None if value is None else str(value),
            )
        row_idx += 1

    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions


wb = Workbook()
default = wb.active
wb.remove(default)

# 1. Scope моделей
model_cols = ["model_sid", "model_code", "model_name"]
add_sheet(
    wb,
    "MODELS",
    model_cols,
    spark_rows(
        models_scope.orderBy("model_sid"),
        model_cols,
    ),
)

# 2. Scope версий
version_cols = versions_scope.columns
add_sheet(
    wb,
    "VERSIONS",
    version_cols,
    spark_rows(
        versions_scope.orderBy("model_sid", "model_ver_sid"),
        version_cols,
    ),
)

# 3. Coverage всех таблиц
coverage_cols = [
    "table_name",
    "status",
    "relation_keys",
    "has_rows",
    "error",
]
add_sheet(
    wb,
    "COVERAGE",
    coverage_cols,
    spark_rows(
        coverage.orderBy("table_name"),
        coverage_cols,
    ),
)

# 4. По 10 значений каждого столбца
values_cols = [
    "table_name",
    "column_name",
    "value_no",
    "value",
]
add_sheet(
    wb,
    "VALUES_10",
    values_cols,
    spark_rows(
        values10.orderBy(
            "table_name",
            "column_name",
            "value_no",
        ) if values10 is not None else None,
        values_cols,
    ) if values10 is not None else [],
)

# 5. Всё подозрительное на ссылки / репозитории
hits_cols = [
    "table_name",
    "column_name",
    "hit_no",
    "value",
]
add_sheet(
    wb,
    "LINK_HITS",
    hits_cols,
    spark_rows(
        link_hits.orderBy(
            "table_name",
            "column_name",
            "hit_no",
        ) if link_hits is not None else None,
        hits_cols,
    ) if link_hits is not None else [],
)

# 6. Полные строки, где найден link-like контент
row_cols = ["table_name", "row_json"]
add_sheet(
    wb,
    "LINK_ROWS",
    row_cols,
    spark_rows(
        link_rows.orderBy("table_name")
        if link_rows is not None else None,
        row_cols,
    ) if link_rows is not None else [],
)

wb.save(OUTPUT_FILE)

print("Готово:", OUTPUT_FILE)
print(
    "Ищи сначала в листе LINK_HITS, затем сверяй строку в LINK_ROWS."
)


## 6. Быстрая диагностика Bitbucket

In [ ]:
# Быстрый просмотр только Bitbucket-подобных значений после выполнения:
if link_hits is not None:
    (
        link_hits
        .filter(F.lower(F.col("value")).contains("bitbucket"))
        .orderBy("table_name", "column_name", "hit_no")
        .show(500, truncate=False)
    )


## Как смотреть результат

После выполнения всех ячеек создаётся `model_mart_probe.xlsx`.

Для поиска ссылки на код:
1. открой `LINK_HITS`;
2. отфильтруй `value` по `bitbucket`;
3. посмотри `table_name` и `column_name`;
4. при необходимости проверь полную исходную строку в `LINK_ROWS`.

`VALUES_10` содержит до 10 различных сырых значений каждого столбца каждой связанной таблицы.